# LangChain Tools & Memory

Tools in LangChain = same concept as function calling
but wrapped in LangChain's tool format.

Memory in LangChain = storing conversation history
so the LLM remembers what was said before.

# Setup

In [9]:
import os
from unittest import result
from urllib import response
from xml.etree.ElementPath import ops

from dotenv import load_dotenv
from langchain_core import messages
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from platformdirs import user_runtime_dir

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

# Defining tools the LangChain way
@tool decorator converts a Python function into a LangChain tool

It uses the docstring as the tool description — write it clearly

In [10]:
@tool
def calculate(operation: str, a: float, b: float):
    """Performs basic math calculations.
    Use this whenever the user asks to calculate, compute or solve math.
    Operations: add, subtract, multiply, divide."""
    ops = {
        "add": a+b,
        "subtract": a-b,
        "multiply": a*b,
        "divide": a/b if b!=0 else "Error! divide by zero"
    }
    return str(ops.get(operation, "Unknown operation"))

@tool
def search_web(query: str):
    """Searches the web for current information.
    Use when user asks about recent events or news."""
    return f"Search results for '{query}': [simulated results about {query}]"

@tool
def get_weather(city: str):
    """Gets current weather for a city.
    Use when user asks about weather or temperature."""
    return f"{city}: 30 degrees Celsius, Sunny"

tools = [calculate, search_web, get_weather]

print("Tools defined:")
for t in tools:
    print(f" - {t.name}: {t.description[:50]}...")

Tools defined:
 - calculate: Performs basic math calculations.
    Use this whe...
 - search_web: Searches the web for current information.
    Use ...
 - get_weather: Gets current weather for a city.
    Use when user...


# Bind tools to LLM

In [11]:
llm_with_tools = llm.bind_tools(tools)

# tool map for execution
tool_map = {t.name: t for t in tools}

# Simple tool calling loop

In [12]:
def run_with_tools(user_input: str):
    messages = [
        SystemMessage(content="You are a helpful assistant. Use tools when needed"),
        HumanMessage(content=user_input)
    ]

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tool_call in response.tool_calls:
            name = tool_call["name"]
            args = tool_call["args"]

            print(f"Calling tool: {name}({args}")
            result = tool_map[name].invoke(args)
            print(f"Result: {result}")

            from langchain_core.messages import ToolMessage
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

print(run_with_tools("What is 234 multiplied by 56?"))
print("---")
print(run_with_tools("What's the weather in Karachi?"))

Calling tool: calculate({'a': 234, 'b': 56, 'operation': 'multiply'}
Result: 13104.0
234 multiplied by 56 equals **13,104**.
---
Calling tool: get_weather({'city': 'Karachi'}
Result: Karachi: 30 degrees Celsius, Sunny
**Karachi Weather:** 30 °C, Sunny.


# Conversation Memory (manual approach)

LangChain doesn't add memory automatically.
You manage the messages list yourself.
This gives you full control.


In [14]:
class ConversationAgent:
    def __init__(self):
        self.llm = ChatGroq(
            model="openai/gpt-oss-20b",
            temperature=0.7,
            api_key=os.getenv("GROQ_API_KEY")
        )
        self.history = [] # for storing conversation

    def chat(self, user_input: str):
        self.history.append(HumanMessage(content=user_input))

        messages = [
            SystemMessage(content="You are a helpful assistant. Remember the conversation context")] + self.history

        response = self.llm.invoke(messages)

        self.history.append(response)

        return response.content

    def clear_history(self):
        self.history = []
        print("History cleared")

    def show_history(self):
        for msg in self.history:
            role = "User" if isinstance(msg, HumanMessage) else "Assistance"
            print(f"{role}: {msg.content[:80]}")

In [15]:
agent = ConversationAgent()

print(agent.chat("My name is Muhammad Usman and I am learning agentic AI."))
print("---")
print(agent.chat("What is my name?"))        # should remember
print("---")
print(agent.chat("What am I learning?"))     # should remember
print("---")
agent.show_history()

Nice to meet you, Muhammad! 👋  
Agentic AI is a fascinating field—essentially about building systems that can act autonomously and make decisions that align with goals or values.  

What specifically are you exploring right now?  
- Reinforcement learning and policy optimization?  
- Goal‑oriented planning or hierarchical agents?  
- Multi‑agent coordination?  
- Ethical alignment and safety?  

Let me know where you’re at, and I can point you toward resources, examples, or even walk through a small project together!
---
Your name is **Muhammad Usman**.
---
You’re learning **agentic AI**—the study of building intelligent agents that can act autonomously, make decisions, and pursue goals or values in a dynamic environment.
---
User: My name is Muhammad Usman and I am learning agentic AI.
Assistance: Nice to meet you, Muhammad! 👋  
Agentic AI is a fascinating field—essentially ab
User: What is my name?
Assistance: Your name is **Muhammad Usman**.
User: What am I learning?
Assistance: You